In [1]:
from pathlib import Path
import pandas as pd

# Replace with your CSV filename
csv_path = Path.cwd() / "XAUUSD Trading Plan.csv"

df = pd.read_csv(csv_path)
df.head()

,Date Open,Date End,Pair,Balance,Risk %,Risk Amount ($),Buy/Sell,Technique,Entry (Price),Stop Loss (Price),Stop Loss (Points),Take Profit (Price),Lot Size,Close Price,Status,Result,New Balance,หมายเหตุ
0,2026-10-22,2026-10-22,XAUUSD,$100.01,10.00%,$10.00,Buy,Liquidity Sweep,"$4,116.11","$4,105.31",1080.0,4125.94,0.01,4105.31,Hit SL,-$10.80,$89.21,แต่ว่ากราฟไปไม่ถึง TP
1,2026-10-22,2026-10-22,XAUUSD,$89.21,10.00%,$8.92,Sell,Trend Follow,"$4,062.47","$4,069.00",653.0,4061.95,0.01,4061.95,Hit TP,$0.52,$89.73,กลัว ปิดหนีไปก่อน
2,2026-10-12,2026-10-12,XAUUSD,$89.73,10.00%,$8.97,Sell,Trend Follow,"$4,058.38","$4,064.33",595.0,4046.09,0.01,4046.09,Hit TP,$12.29,$102.02,ชน TP
3,2026-10-13,2026-10-13,XAUUSD,$102.02,10.00%,$10.20,Sell,Own Predict,"$4,040.87","$4,050.60",973.0,4030.34,0.01,4045.41,Manual,-$4.54,$97.48,เข้าแบบยังไม่ครบเงื่อนไขของ Setup
4,2026-10-13,2026-10-13,XAUUSD,$97.48,10.00%,$9.75,Buy,Liquidity Sweep,"$4,045.56","$4,034.02",1154.0,4059.24,0.01,4034.02,Hit SL,-$11.54,$85.94,เล่น LQ Sweep ใน M1 จาก Zone ที่พิจารณาใน HTF ...


In [10]:
# Exclude "Own Predict" trades and empty rows
filtered_df = df[
    df["Technique"].notna()
    & df["Technique"].ne("Own Predict")
].copy()

# Calculate win rate using only Hit TP and Hit SL trades
completed_trades = filtered_df[
    filtered_df["Status"].isin(["Hit TP", "Hit SL"])
]

winrate = (
    completed_trades["Status"].eq("Hit TP").mean() * 100
    if not completed_trades.empty
    else 0
)

raw = df
raw_completed_trades = raw[
    raw["Status"].isin(["Hit TP", "Hit SL"])
]

windrate_raw = (
    raw_completed_trades["Status"].eq("Hit TP").mean() * 100
    if not raw_completed_trades.empty
    else 0
)

print(f"Raw win rate: {windrate_raw:.2f}%")
print(f"Filtered win rate: {winrate:.2f}%")

Raw win rate: 40.00%
Filtered win rate: 66.67%


## Own Predict — เกิดอะไรขึ้น + Sentiment analysis

ใช้ทั้งไฟล์ (ทุก Technique) เพื่อเทียบ Own Predict กับเทรดที่มี Setup:

1. **ผลลัพธ์** — Own Predict vs Setup trades และพอร์ตถ้าไม่มี Own Predict
2. **บริบท** — เข้า Own Predict หลังจากเทรดก่อนหน้าเป็นยังไง, สลับฝั่ง Buy/Sell, กระจุกวันไหน
3. **Sentiment ของหมายเหตุ** — lexicon คำ/วลีจากบันทึกนี้เอง (ไม่มี Thai NLP library และ model ทั่วไปอ่านศัพท์เทรดไม่ออก) ให้คะแนน + ติดป้ายสัญญาณ เช่น ผิดกฎ Setup, ไม่มั่นใจ, รีบ / FOMO

In [11]:
# Lexicon built from this journal's own wording. Substring match (Thai has no spaces),
# cue -> (signal, weight). Negative = the note admits a problem with the trade.
LEXICON = {
    "ไม่ครบ": ("ผิดกฎ Setup", -2),
    "ไม่มี confirm": ("ผิดกฎ Setup", -2),
    "ยังไม่ได้ clear": ("ผิดกฎ Setup", -1),
    "ไม่ชัวร์": ("ไม่มั่นใจ", -1),
    "เหมือนจะ": ("ไม่มั่นใจ", -1),
    "รีบเข้า": ("รีบ / FOMO", -2),
    "ก็เลยกด": ("รีบ / FOMO", -1),
    "ติดลบ": ("ตอบโต้ order ที่ขาดทุน", -2),
    "ไม่ได้ดู": ("วิเคราะห์ไม่ครบ", -1),
    "tp เกิน": ("วิเคราะห์ไม่ครบ", -1),
    "แต่ bias htf": ("สวน Bias ใหญ่", -1),
    "กลัว": ("กลัว", -2),
    "ปิดหนี": ("กลัว", -1),
    "ไปไม่ถึง": ("ผิดหวัง", -1),
    "เสมอตัว": ("ปิดเท่าทุน", 0),
    "เท่าทุน": ("ปิดเท่าทุน", 0),
    "ชน tp": ("ตามแผน", 2),
    "เอาอยู่": ("มี Confirm", 1),
    "ถูกปฎิเสธราคา": ("มี Confirm", 1),
    "break โครงสร้าง": ("อ่านโครงสร้าง", 1),
    "breaker block": ("อ่านโครงสร้าง", 1),
    "mss": ("อ่านโครงสร้าง", 1),
    "kl": ("อ่านโครงสร้าง", 1),
}

def sentiment(note):
    text = note.lower()
    hits = [(cat, w) for cue, (cat, w) in LEXICON.items() if cue in text]
    score = sum(w for _, w in hits)
    if not note:
        label = "ไม่ได้จด"
    else:
        label = "ลบ" if score < 0 else "บวก" if score > 0 else "กลาง"
    return pd.Series({
        "score": score,
        "sentiment": label,
        "signals": ", ".join(dict.fromkeys(cat for cat, _ in hits)),
    })

trades = trades.drop(columns=["score", "sentiment", "signals"], errors="ignore")
trades = trades.join(trades["note"].apply(sentiment))
op = trades[trades["is_op"]]
rest = trades[~trades["is_op"]]

NameError: name 'trades' is not defined

In [ ]:
pd.set_option("display.max_colwidth", None)

# Every note scored and labelled, then compared by group
op_cols = ["seq", "Date Open", "Buy/Sell", "Status", "pnl", "sentiment", "score", "signals", "note"]
display(op[op_cols])

display(pd.crosstab(trades["Technique"].where(trades["is_op"], "Setup trades"), trades["sentiment"]))

# Does the note's sentiment line up with the result, whatever the label?
display(trades.groupby("sentiment")["pnl"].agg(["size", "sum", "mean"]).round(2))

# Most common signals in Own Predict notes
display(op["signals"].str.split(", ").explode().replace("", pd.NA).dropna().value_counts())

# Setup trades whose own note is negative -- the same problems under a setup label
display(rest[rest["score"] < 0][["seq", "Technique", "pnl", "signals", "note"]])